# Risk-group discovery: causal candidates vs other group finders

**One notebook, two datasets, the same rule.** Set `DATASET` in section 0:
* `"cdc"` — CDC BRFSS diabetes indicators, full survey (the large dataset; saved as notebook 06).
* `"nhanes"` — NHANES 2005-2018 (the dataset with **real ground truth**: self-reported diagnosis vs a
  blood test; saved as notebook 08). Run notebook 07 first.

**What this answers.** Can causal discovery, run on the *training data only* and *before any predictive model is
trained*, point to the subgroups that carry risk — and how does it compare with the other ways people
look for such groups (slice search, clustering, confident learning, KNN label disagreement, anomaly
detection, a protected-attributes-only fairness check, random groups)?

**Two kinds of risk are checked.** (a) *Model risk*: the trained model does worse on the group, judged on
the locked test set (section 6). (b) *Recording risk*: the group's outcomes are under-recorded in the
labels themselves (section 4; and, for NHANES only, checked against the blood-test truth in section 9).

**No noise is injected and nothing is repaired.** Every method gets the same training data; nothing in
sections 1-5 looks at the test set; and nothing before section 9 looks at the NHANES blood-test truth.

| Section | Output (`NB` = 06 for CDC, 08 for NHANES) |
|---|---|
| 1 Data and split | `data/cdcfull_*.csv` (CDC) or reads `data/nhanes_*.csv` (from notebook 07) |
| 2 Stage 0 | `results/NB_stage0_edges_*.csv`, `results/NB_stage0_gate_*.json` |
| 3 Causal candidates (the proposed rule) | `results/NB_candidates_causal.csv` |
| 4 Recording-risk check (pre-training) | `results/NB_observed_expected.csv` |
| 5 Baseline group finders | `results/NB_candidates_all.csv`, `results/NB_rankings.csv` |
| 6 Train once, confirm on test | `results/NB_test_confirmation.csv` |
| 7 Scorecard | `results/NB_scorecard.csv` |
| 8 Blind spot: recording-path groups | `results/NB_blind_spot.csv` |
| 9 Ground truth (NHANES only) | `results/NB_ground_truth*.csv` (all groups, scorecard, answer key, short lists) |

## 0. Settings
`QUICK = True` is only for checking that the notebook runs (small sample, few bootstraps). Use
`QUICK = False` for any number that goes in the paper.

In [12]:
QUICK = False

import sys, time, json, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

import li_lib as li
print("li_lib", li.VERSION, "| root:", li.ROOT)

DATASET = "nhanes"      # "cdc" or "nhanes"
REUSE_STAGE0 = True     # True: if this dataset's Stage 0 output already exists, load it instead of refitting

SEED = 20261020
if DATASET == "cdc":
    NB, TAG = "06", "cdcfull"
    LABEL, TRUTH = "diabetes_any", None
    FEATURES = list(li.CDC_DIABETES.numeric)             # the 21 survey indicators
    PROTECTED = ["Sex", "Age"]
else:
    NB, TAG = "08", "nhanes"
    LABEL, TRUTH = "diabetes_recorded", "diabetes_true"  # TRUTH is used in section 9 only
    FEATURES = ["HighBP", "HighChol", "CholCheck", "AnyHealthcare", "RoutineCare", "VisitedDoctor", "BMI",
                "Smoker", "Stroke", "HeartDiseaseorAttack", "GenHlth", "Sex", "Age", "Education", "Income",
                "race_black", "race_hispanic"]
    PROTECTED = ["Sex", "Age", "race_black", "race_hispanic"]
RECORDING = ["CholCheck"]              # declared, same in both datasets: a cholesterol check in the past
                                       # 5 years measures contact with care. It is how the outcome gets
                                       # *recorded*, not what causes it. Its edge into the label is read as
                                       # a recording edge.
MIN_SHARE, MAX_SHARE = 0.03, 0.50      # a group must be 3%-50% of the training data
MIN_N_TRAIN = 500                      # and hold at least 500 people. Deliberately NOT a minimum number of
                                       # recorded positives: an under-recorded group has few of them.
EDGE_P = 0.80                          # an edge is "stable" if it appears in >= 80% of bootstraps
MIN_EFFECT = 0.01                      # |coefficient| below this = no edge (same as Stage 0)
BOOT = 10 if QUICK else 50
MAX_N_DISCOVERY = 8000 if QUICK else 20000
N_ROWS = 40000 if QUICK else None      # CDC only. None = the full de-duplicated survey
ALPHA = 0.05
OUT = li.RESULTS / "_quick" if QUICK else li.RESULTS   # quick runs never overwrite paper results
OUT.mkdir(exist_ok=True)
def save(d, name, show=0):
    return li.save(d, name, folder=OUT, show=show)
print("dataset:", DATASET, "| outputs prefixed", NB)

li_lib 1.1.0 | root: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity
dataset: nhanes | outputs prefixed 08


## 1. Data and split (60 / 20 / 20)
Stratified on the recorded label only — **not** on any group, because the risk group is not known in
advance. NHANES was split the same way in notebook 07.

In [13]:
from sklearn.model_selection import train_test_split
if DATASET == "cdc":
    raw = pd.read_csv(li.DATA / "cdc_diabetes_raw.csv")
    df = raw.copy()
    df[LABEL] = (df["Diabetes_binary"] > 0).astype(int)
    df = df.drop(columns=["Diabetes_binary"]).dropna().reset_index(drop=True)
    if N_ROWS and len(df) > N_ROWS:
        df = df.sample(N_ROWS, random_state=SEED).reset_index(drop=True)
    train, rest = train_test_split(df, test_size=0.4, random_state=SEED, stratify=df[LABEL])
    val, test = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest[LABEL])
    train, val, test = (d.reset_index(drop=True) for d in (train, val, test))
    if not QUICK:
        for name, d in [("train", train), ("val", val), ("test", test)]:
            d.to_csv(li.DATA / f"{TAG}_{name}.csv", index=False)
    print(f"raw rows {len(raw):,} -> de-duplicated {len(df):,}")
else:
    train, val, test = (pd.read_csv(li.DATA / f"nhanes_{p}.csv") for p in ("train", "val", "test"))
const = [c for c in FEATURES if train[c].nunique() < 2]
if const:   # a constant column breaks causal discovery (divide-by-zero) and carries no information
    print("dropping constant feature(s):", const)
    FEATURES = [c for c in FEATURES if c not in const]
print(pd.DataFrame({"rows": [len(train), len(val), len(test)],
                    "recorded_positive_rate": [train[LABEL].mean(), val[LABEL].mean(), test[LABEL].mean()]},
                   index=["train", "val", "test"]).round(4))

        rows  recorded_positive_rate
train  18463                  0.1283
val     6155                  0.1284
test    6155                  0.1282


## 2. Stage 0 on the training split
Same settings as the September Stage 0 script: DirectLiNGAM with prior knowledge (protected attributes
have no causes, the label causes nothing), a seeded bootstrap, the PC skeleton cross-check, and the
assumption gate. Runs on a seeded sample of `MAX_N_DISCOVERY` training rows. With `REUSE_STAGE0 = True`
a saved run for this dataset is loaded instead (same numbers, no refit).

In [14]:
spec = importlib.util.spec_from_file_location("stage0_diagnose", li.STAGE0_SCRIPT)
s0 = importlib.util.module_from_spec(spec); sys.modules["stage0_diagnose"] = s0; spec.loader.exec_module(s0)

cols = [LABEL] + FEATURES
disc = train[cols].astype(float)
if len(disc) > MAX_N_DISCOVERY:
    disc = disc.sample(MAX_N_DISCOVERY, random_state=SEED)
X_raw = disc.to_numpy(float); X_std = s0.zscore(X_raw)
p = len(cols); yi = cols.index(LABEL); ai = [cols.index(a) for a in PROTECTED]
binary_mask = np.array([s0.is_binary_col(X_raw[:, j]) for j in range(p)])
pk = s0.build_prior_knowledge(p, exogenous=ai, sinks=[yi])

edges_path, gate_path = OUT / f"{NB}_stage0_edges_{TAG}.csv", OUT / f"{NB}_stage0_gate_{TAG}.json"
t0 = time.time()
B_hat, order = s0.fit_directlingam(X_std, pk, SEED)          # point fit (one fit; needed for the gate)
if REUSE_STAGE0 and edges_path.exists():
    edges = pd.read_csv(edges_path)
    edge_prob, adj_prob = np.zeros((p, p)), np.zeros((p, p))
    for f_, t_, pd_, pa_ in zip(edges["from"], edges["to"], edges["prob_directed"], edges["prob_adjacent"]):
        edge_prob[cols.index(t_), cols.index(f_)] = pd_
        adj_prob[cols.index(t_), cols.index(f_)] = pa_
    print(f"loaded saved bootstrap edges ({edges_path.name})")
else:
    adjs, _ = s0.seeded_bootstrap(X_std, pk, BOOT, SEED)
    present = np.abs(adjs) > MIN_EFFECT                      # [b, to, from]
    edge_prob = present.mean(axis=0)                         # directed: from j into i
    adj_prob = (present | present.transpose(0, 2, 1)).mean(axis=0)   # either direction
    edges = pd.DataFrame([{"from": cols[j], "to": cols[i], "prob_directed": edge_prob[i, j],
                           "prob_adjacent": adj_prob[i, j], "coef_median": float(np.median(adjs[:, i, j]))}
                          for i in range(p) for j in range(p) if i != j and adj_prob[i, j] > 0])
    save(edges.sort_values("prob_adjacent", ascending=False), edges_path.name)
try:
    pc_adj = s0.pc_skeleton(X_std, 0.01)
    jac = s0.skeleton_jaccard((np.abs(B_hat) > MIN_EFFECT) | (np.abs(B_hat.T) > MIN_EFFECT), pc_adj)
except Exception as e:
    print("PC skipped:", e); jac = float("nan")
E = X_std - X_std @ B_hat.T
gate = s0.assumption_gate(len(train), p, 2000, float(binary_mask.mean()),
                          s0.nongauss_fraction(E, binary_mask, seed=SEED),
                          s0.nonlinear_edge_fraction(X_std, B_hat, binary_mask),
                          s0.varsortability(X_raw, B_hat, MIN_EFFECT), s0.r2sortability(X_std, B_hat, MIN_EFFECT),
                          jac, [cols[a] for a in ai if binary_mask[a]], has_label=True)
gate_path.write_text(json.dumps({"rung": gate.rung, "reasons": gate.reasons, "warnings": gate.warnings,
                                 "n_train": len(train), "boot": BOOT, "max_n": MAX_N_DISCOVERY}, indent=2))
print(f"Stage 0 took {time.time() - t0:.0f}s | gate: {gate.rung} | {gate.reasons}")

loaded saved bootstrap edges (08_stage0_edges_nhanes.csv)
Stage 0 took 7s | gate: BOUNDED | ['discrete-heavy: 72% of variables binary (linear-continuous SEM misspecified)']


## 3. Causal candidate groups (the proposed method)
**The rule, fixed before looking at any results:**
1. Which edges count depends on the gate. `POINT`: use directed edges. `BOUNDED`: use adjacency in
   either direction, because the gate says the direction of edges between binary variables cannot be
   trusted. `REFUSED`: use the protected attributes only.
2. Candidate attributes = protected attributes + the declared recording variables + attributes with a
   stable edge to a recording variable + attributes with a stable edge to the label. An attribute can
   qualify for several reasons; all are kept (the `why` column).
3. Each attribute becomes groups: each level of a binary attribute; the lower and upper tails
   (<= 20th, >= 80th percentile) of an ordered one. Keep groups holding 3%-50% of the training data and
   at least 500 people (no minimum on recorded positives — an under-recorded group has few).

In [15]:
P = edge_prob if gate.rung == "POINT" else adj_prob
def stable_neighbors(var):
    i = cols.index(var)
    return {cols[j] for j in range(p) if j != i and P[i, j] >= EDGE_P}   # j -> i (or adjacent)

cand_attrs = {}
def tag(a, why):
    cand_attrs.setdefault(a, [])
    if why not in cand_attrs[a]:
        cand_attrs[a].append(why)
for a in PROTECTED: tag(a, "protected")
if gate.rung != "REFUSED":
    for r in RECORDING:
        tag(r, "recording variable")
        for a in sorted(stable_neighbors(r) - {LABEL}): tag(a, "edge to recording")
    for a in sorted(stable_neighbors(LABEL) - set(RECORDING)): tag(a, "edge to label")

def groups_for(attr, d):
    v = d[attr]
    if v.nunique() <= 2:
        return [(f"{attr} == {int(x)}", f"`{attr}` == {x}") for x in sorted(v.unique())]
    lo, hi = v.quantile(0.2), v.quantile(0.8)
    return [(f"{attr} <= {lo:g}", f"`{attr}` <= {lo}"), (f"{attr} >= {hi:g}", f"`{attr}` >= {hi}")]

def keep(mask, y):
    return MIN_SHARE <= mask.mean() <= MAX_SHARE and mask.sum() >= MIN_N_TRAIN

y_tr = train[LABEL].to_numpy()
causal = []
for a, whys in cand_attrs.items():
    for name, q in groups_for(a, train):
        m = train.eval(q).to_numpy(bool)
        if keep(m, y_tr):
            causal.append({"method": "causal (proposed)", "group": name, "query": q, "why": "; ".join(whys),
                           "share_train": m.mean(), "pos_rate_train": y_tr[m].mean()})
causal = pd.DataFrame(causal)
K = len(causal)
print(f"{len(cand_attrs)} candidate attributes -> K = {K} candidate groups")
save(causal, f"{NB}_candidates_causal.csv", show=K)

17 candidate attributes -> K = 22 candidate groups
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_candidates_causal.csv  (22 rows x 6 cols)


,method,group,query,why,share_train,pos_rate_train
0,causal (proposed),Sex == 1,`Sex` == 1.0,protected; edge to label,0.495315,0.135484
1,causal (proposed),Age <= 32,`Age` <= 32.0,protected; edge to recording; edge to label,0.207496,0.012790
2,causal (proposed),Age >= 67,`Age` >= 67.0,protected; edge to recording; edge to label,0.203921,0.245153
3,causal (proposed),race_black == 1,`race_black` == 1.0,protected; edge to label,0.209554,0.158439
4,causal (proposed),race_hispanic == 1,`race_hispanic` == 1.0,protected; edge to recording; edge to label,0.239885,0.146083
5,causal (proposed),CholCheck == 0,`CholCheck` == 0.0,recording variable,0.282944,0.038285
6,causal (proposed),AnyHealthcare == 0,`AnyHealthcare` == 0.0,edge to recording,0.206954,0.078252
7,causal (proposed),BMI <= 23.56,`BMI` <= 23.56,edge to recording; edge to label,0.200130,0.051150
8,causal (proposed),BMI >= 34,`BMI` >= 34.0,edge to recording; edge to label,0.201484,0.222581
9,causal (proposed),Education <= 2,`Education` <= 2.0,edge to recording; edge to label,0.234415,0.186229


WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/label_integrity/results/08_candidates_causal.csv')

## 4. Recording-risk check (pre-training; the label-integrity part)
**Question:** does a group have *fewer recorded outcomes than its risk factors predict*? That is what
under-recording (here: undiagnosed diabetes) looks like.

**How:** fit a model on everyone **outside** the group, leaving out the recording variable(s) and the
attribute that defines the group, and predict how many recorded positives the group *should* have.
(Keeping the recording variable in would "explain away" the very gap we are looking for.) Observed / expected (O/E) well below 1 is
the warning sign. Exact Poisson 95% interval on the observed count.

**When the check is valid — this is where the causal graph is needed.** Leaving an attribute out is only
fair if it has no stable causal edge into the outcome; otherwise O/E < 1 just means "this attribute
really lowers risk". And the group's other risk factors must overlap with the rest of the data; the tails
of a continuous variable (BMI <= 23, youngest ages) fail this, because the outside model has never seen
such people. So the check is marked `valid` only for groups defined by a recording variable, or by an
attribute with a stable edge to a recording variable and **no** stable edge to the label.

**Limit, stated plainly:** the data alone cannot separate "truly lower risk" from "missed diagnoses", and
the test set cannot settle it, because its labels are recorded the same way. O/E < 1 is a flag for
verification, not a verdict.

In [16]:
def observed_expected(mask, attr):
    feats = [f for f in FEATURES if f not in set(RECORDING) | {attr}]
    clf = li.fit_model_b(train.loc[~mask, feats].to_numpy(float), y_tr[~mask], np.ones((~mask).sum()), SEED)
    e = clf.predict_proba(train.loc[mask, feats].to_numpy(float))[:, 1].sum()
    o = y_tr[mask].sum()
    lo, hi = stats.chi2.ppf(0.025, 2 * o) / 2, stats.chi2.ppf(0.975, 2 * (o + 1)) / 2   # exact Poisson CI on O
    return o, e, o / e, lo / e, hi / e

def oe_valid(attr):
    whys = cand_attrs.get(attr, [])
    return "recording variable" in whys or ("edge to recording" in whys and "edge to label" not in whys)

oe = []
for r in causal.itertuples():
    attr = r.group.split()[0]
    m = train.eval(r.query).to_numpy(bool)
    o, e, ratio, lo, hi = observed_expected(m, attr)
    oe.append({"group": r.group, "why": r.why, "valid": oe_valid(attr), "observed": int(o), "expected": round(e, 1),
               "obs_over_exp": ratio, "ci_low": lo, "ci_high": hi,
               "flag_under_recorded": bool(oe_valid(attr) and hi < 1)})
oe = pd.DataFrame(oe).sort_values(["valid", "obs_over_exp"], ascending=[False, True])
save(oe, f"{NB}_observed_expected.csv", show=K)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_observed_expected.csv  (22 rows x 9 cols)


,group,why,valid,observed,expected,obs_over_exp,ci_low,ci_high,flag_under_recorded
5,CholCheck == 0,recording variable,True,200,275.399994,0.726344,0.629163,0.834286,True
15,RoutineCare == 0,edge to recording,True,103,139.300003,0.739205,0.603363,0.896500,True
16,Smoker == 1,edge to recording,True,1183,1352.199951,0.874842,0.825694,0.926152,True
14,Income >= 4.8,edge to recording,True,343,364.500000,0.941118,0.844145,1.046176,False
13,Income <= 0.97,edge to recording,True,548,543.599976,1.008152,0.925500,1.096206,False
6,AnyHealthcare == 0,edge to recording,True,299,270.500000,1.105310,0.983563,1.237964,False
1,Age <= 32,protected; edge to recording; edge to label,False,49,223.000000,0.219714,0.162546,0.290474,False
17,VisitedDoctor == 0,edge to recording; edge to label,False,69,189.000000,0.365053,0.284033,0.461997,False
18,GenHlth <= 2,edge to label,False,324,773.000000,0.419123,0.374723,0.467337,False
7,BMI <= 23.56,edge to recording; edge to label,False,189,327.500000,0.577172,0.497816,0.665582,False


WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/label_integrity/results/08_observed_expected.csv')

## 5. Baseline group finders (all on the training split, each proposes K groups)
* **Protected-only fairness check:** groups from Sex and Age only (the conventional audit).
* **Slice search (SliceLine-style):** every single-attribute group and every pair of them, ranked by how
  many positives an out-of-fold model misses in the group compared with the rest.
* **Clustering:** k-means clusters, ranked the same way.
* **Confident learning concentration:** single-attribute groups ranked by the share of records cleanlab flags.
* **KNN label disagreement (SimiFeat-style):** single-attribute groups ranked by the share of records whose
  label disagrees with the majority of their 10 nearest neighbors.
* **Anomaly detection:** single-attribute groups ranked by the share of isolation-forest anomalies (top 5%).
* **Random:** K single-attribute groups drawn at random, repeated 20 times.

Every method's list is de-duplicated the same way: a group is skipped if more than 70% of the smaller of
the two groups lies inside a group already on that method's list (so `BMI <= 23 & Smoker == 1` does not
count as a new find next to `BMI <= 23`).

In [17]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest

X_tr = train[FEATURES].to_numpy(float)
sc_all = StandardScaler().fit(X_tr); Z_tr = sc_all.transform(X_tr)
rate = y_tr.mean()

# out-of-fold scores from the same model family used in section 6
oof = np.zeros(len(train))
for k, (a, b) in enumerate(StratifiedKFold(3 if QUICK else 5, shuffle=True, random_state=SEED).split(X_tr, y_tr)):
    oof[b] = li.fit_model_b(X_tr[a], y_tr[a], np.ones(len(a)), SEED + k).predict_proba(X_tr[b])[:, 1]
oof_pred = oof >= np.quantile(oof, 1 - rate)

def missed_gap(mask):
    pos = y_tr == 1
    return oof_pred[pos & ~mask].mean() - oof_pred[pos & mask].mean()   # rest recall - group recall

# universe of single-attribute groups (all 21 indicators, same construction as section 3)
singles = []
for a in FEATURES:
    for name, q in groups_for(a, train):
        m = train.eval(q).to_numpy(bool)
        if keep(m, y_tr):
            singles.append((name, q, m))
pairs = []
for i in range(len(singles)):
    for j in range(i + 1, len(singles)):
        if singles[i][0].split()[0] == singles[j][0].split()[0]:
            continue
        m = singles[i][2] & singles[j][2]
        if keep(m, y_tr):
            pairs.append((f"{singles[i][0]} & {singles[j][0]}", f"({singles[i][1]}) & ({singles[j][1]})", m))
print(f"slice universe: {len(singles)} single-attribute groups, {len(pairs)} pairs")

RANKS = {}   # each method's full ranking of the groups it scored (used in section 8)
def top_k(method, scored, k=K):
    """scored: list of (name, query, mask, score). Highest score first, de-duplicated."""
    if not method.startswith("random"):
        RANKS[method] = [n for n, q, m, s in sorted(scored, key=lambda t: -t[3])]
    chosen = []
    for name, q, m, s in sorted(scored, key=lambda t: -t[3]):
        if any((m & c[2]).sum() / min(m.sum(), c[2].sum()) > 0.7 for c in chosen):
            continue
        chosen.append((name, q, m, s))
        if len(chosen) == k:
            break
    return [{"method": method, "group": n, "query": q, "score_train": s,
             "share_train": m.mean(), "pos_rate_train": y_tr[m].mean()} for n, q, m, s in chosen]

rows = []
# protected only
prot = [(n, q, m, 0.0) for n, q, m in singles if n.split()[0] in PROTECTED]
rows += top_k("protected only", prot)
# slice search
rows += top_k("slice search", [(n, q, m, missed_gap(m)) for n, q, m in singles + pairs])
# clustering (cluster id stored as the query; resolved in section 6 with km.predict)
km = KMeans(n_clusters=10 if QUICK else 20, n_init=5, random_state=SEED).fit(Z_tr)
cl_scored = [(f"cluster {c}", f"__cluster__{c}", km.labels_ == c, missed_gap(km.labels_ == c))
             for c in range(km.n_clusters) if keep(km.labels_ == c, y_tr)]
rows += top_k("clustering", cl_scored)
# confident learning
from cleanlab.filter import find_label_issues
cl_flags = find_label_issues(y_tr, np.c_[1 - oof, oof])
rows += top_k("confident learning", [(n, q, m, cl_flags[m].mean()) for n, q, m in singles])
# KNN label disagreement
ref = np.random.default_rng(SEED).choice(len(train), min(len(train), 50000), replace=False)
nn = NearestNeighbors(n_neighbors=11).fit(Z_tr[ref])
idx = nn.kneighbors(Z_tr, return_distance=False)[:, 1:]          # drop the nearest (often itself)
knn_flag = (y_tr[ref][idx].mean(axis=1) >= 0.5).astype(int) != y_tr
rows += top_k("KNN disagreement", [(n, q, m, knn_flag[m].mean()) for n, q, m in singles])
# anomaly detection
iso = IsolationForest(random_state=SEED, n_jobs=li.N_JOBS).fit(Z_tr)
anom = -iso.score_samples(Z_tr); anom_flag = anom >= np.quantile(anom, 0.95)
rows += top_k("anomaly detection", [(n, q, m, anom_flag[m].mean()) for n, q, m in singles])
# random (20 draws)
rng = np.random.default_rng(SEED)
for r in range(20):
    pick = [singles[i] for i in rng.permutation(len(singles))]
    rows += [dict(x, method=f"random#{r}") for x in top_k("random", [(n, q, m, 0.0) for n, q, m in pick])]

cands = pd.concat([causal.assign(score_train=np.nan), pd.DataFrame(rows)], ignore_index=True)
save(cands.drop(columns=["why"], errors="ignore"), f"{NB}_candidates_all.csv", show=0)
save(pd.DataFrame([{"method": k, "rank": i + 1, "group": g} for k, v in RANKS.items() for i, g in enumerate(v)]),
     f"{NB}_rankings.csv", show=0)
print(cands[~cands.method.str.startswith("random#")].groupby("method").size())

slice universe: 22 single-attribute groups, 162 pairs
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_candidates_all.csv  (543 rows x 6 cols)
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_rankings.csv  (273 rows x 3 cols)
method
KNN disagreement      21
anomaly detection     21
causal (proposed)     22
clustering            18
confident learning    21
protected only         5
slice search          22
dtype: int64


## 6. Train the model once, then confirm every candidate on the locked test set
One XGBoost model is trained on the training split (the same model as notebooks 03-04). The threshold
flags the top share of validation scores equal to the training positive rate. Each candidate group is
checked on the **test** set two ways, each a one-sided test, Holm-corrected within each method's K groups:

1. **Missed positives** — does the model catch fewer of this group's recorded positives than everyone
   else's? (Fisher exact.) *Caution:* at one global threshold, groups with a low base rate (young, lean,
   healthy) always score worse here, even when the model ranks them correctly. Read it next to group AUC.
2. **Under-prediction** — does the group have more recorded positives than the model predicts?
   (z-test of observed vs the sum of predicted probabilities.) This is the calibration harm.

Both tests use the *recorded* test labels. A group whose labels are under-recorded the same way in
training and test will look fine here — that is the blind spot section 4 is for.

In [18]:
model = li.fit_model_b(X_tr, y_tr, np.ones(len(y_tr)), SEED)
p_val = model.predict_proba(val[FEATURES].to_numpy(float))[:, 1]
thr = li.threshold_top_rate(p_val, rate)
X_te = test[FEATURES].to_numpy(float); y_te = test[LABEL].to_numpy()
p_te = model.predict_proba(X_te)[:, 1]; pred_te = p_te >= thr
clu_te = km.predict(sc_all.transform(X_te))
overall = li.evaluate(y_te, p_te, np.zeros(len(y_te), bool), thr)
print(f"overall test AUC {overall['auc']:.3f} | recall {overall['recall']:.3f} | ECE {overall['ece']:.3f}")

def test_mask(q):
    return clu_te == int(q.split("__")[-1]) if q.startswith("__cluster__") else test.eval(q).to_numpy(bool)

def confirm(q):
    m = test_mask(q); pos = y_te == 1
    a, b = pred_te[pos & m].sum(), (~pred_te[pos & m]).sum()      # group: caught, missed
    c, d = pred_te[pos & ~m].sum(), (~pred_te[pos & ~m]).sum()    # rest:  caught, missed
    pval = stats.fisher_exact([[a, b], [c, d]], alternative="less")[1] if (a + b) else np.nan
    return {"n_test": int(m.sum()), "pos_test": int(a + b),
            "recall_group": a / max(a + b, 1), "recall_rest": c / max(c + d, 1),
            "recall_gap": c / max(c + d, 1) - a / max(a + b, 1),
            "calib_ratio": p_te[m].mean() / max(y_te[m].mean(), 1e-9),
            "ece_group": li.expected_calibration_error(y_te[m], p_te[m]),
            "auc_group": auc(m), "auc_rest": auc(~m),
            "p_missed": pval, "p_underpred": underpred(m)}

from sklearn.metrics import roc_auc_score
def auc(m):
    return roc_auc_score(y_te[m], p_te[m]) if len(np.unique(y_te[m])) == 2 else np.nan

def underpred(m):
    o, e, v = y_te[m].sum(), p_te[m].sum(), (p_te[m] * (1 - p_te[m])).sum()
    return float(stats.norm.sf((o - e) / np.sqrt(v))) if v > 0 else np.nan

conf = pd.concat([cands, pd.DataFrame([confirm(q) for q in cands["query"]])], axis=1)
from statsmodels.stats.multitest import multipletests
for test_name in ["missed", "underpred"]:
    col = f"p_{test_name}"; conf[f"{col}_holm"] = np.nan
    for mth, g in conf.groupby("method"):
        ok = g[col].notna()
        conf.loc[g.index[ok], f"{col}_holm"] = multipletests(g.loc[ok, col], method="holm")[1]
    conf[f"confirmed_{test_name}"] = conf[f"{col}_holm"] < ALPHA
save(conf.drop(columns=["why"], errors="ignore"), f"{NB}_test_confirmation.csv", show=0)
show = ["method", "group", "share_train", "n_test", "pos_test", "recall_group", "recall_rest",
        "recall_gap", "auc_group", "auc_rest", "calib_ratio", "confirmed_missed", "confirmed_underpred"]
conf.loc[~conf.method.str.startswith("random#"), show].round(3)

overall test AUC 0.839 | recall 0.422 | ECE 0.010
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_test_confirmation.csv  (543 rows x 21 cols)


,method,group,share_train,n_test,pos_test,recall_group,recall_rest,recall_gap,auc_group,auc_rest,calib_ratio,confirmed_missed,confirmed_underpred
0,causal (proposed),Sex == 1,0.495,3019,393,0.461,0.384,-0.077,0.851,0.827,1.024,False,False
1,causal (proposed),Age <= 32,0.207,1286,24,0.000,0.435,0.435,0.768,0.811,0.682,True,False
2,causal (proposed),Age >= 67,0.204,1274,297,0.428,0.419,-0.009,0.705,0.861,1.029,False,False
3,causal (proposed),race_black == 1,0.210,1359,216,0.500,0.393,-0.107,0.831,0.839,1.010,False,False
4,causal (proposed),race_hispanic == 1,0.240,1507,216,0.486,0.398,-0.088,0.823,0.845,1.003,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,anomaly detection,Age <= 32,0.207,1286,24,0.000,0.435,0.435,0.768,0.811,0.682,True,False
126,anomaly detection,BMI <= 23.56,0.200,1216,60,0.033,0.454,0.421,0.826,0.829,1.054,True,False
127,anomaly detection,GenHlth <= 2,0.398,2431,105,0.000,0.487,0.487,0.834,0.795,0.962,True,False
128,anomaly detection,Education >= 5,0.239,1358,115,0.278,0.447,0.168,0.850,0.832,0.999,True,False


## 7. Scorecard
* **confirmed_missed / confirmed_underpred** — how many of a method's K groups pass each test-set check
  (on data the method never saw); **share_confirmed_any** — share passing either (higher = fewer false leads).
* **median gap (confirmed)** — size of the missed-positive shortfall in the groups that passed.
* **min group AUC / max calibration shortfall** — the worst ranking and the worst under-prediction found.
* **needs a trained model** — whether the method can run before training.
* **interpretable rule** — whether the group comes as a rule a person can act on.
Random is averaged over its 20 draws.

In [19]:
def card(g, name):
    cm, cu = g[g.confirmed_missed], g[g.confirmed_underpred]
    return {"method": name, "K": len(g),
            "confirmed_missed": int(len(cm)), "confirmed_underpred": int(len(cu)),
            "share_confirmed_any": (g.confirmed_missed | g.confirmed_underpred).mean() if len(g) else np.nan,
            "median_gap_confirmed": cm.recall_gap.median() if len(cm) else np.nan,
            "min_auc_group": g.auc_group.min(), "max_calib_shortfall": 1 - g.calib_ratio.min()}
order = ["causal (proposed)", "protected only", "slice search", "clustering", "confident learning",
         "KNN disagreement", "anomaly detection"]
sc_rows = [card(conf[conf.method == m], m) for m in order]
rnd = pd.DataFrame([card(conf[conf.method == f"random#{r}"], "random") for r in range(20)])
sc_rows.append({"method": "random (mean of 20)", **rnd.drop(columns="method").mean().to_dict()})
score = pd.DataFrame(sc_rows)
score["needs_trained_model"] = score.method.isin(["slice search", "clustering", "confident learning"])
score["interpretable_rule"] = score.method != "clustering"
save(score, f"{NB}_scorecard.csv", show=0)
print(f"overall test AUC {overall['auc']:.3f}, overall recall {overall['recall']:.3f}")
score.round(3)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_scorecard.csv  (8 rows x 10 cols)
overall test AUC 0.839, overall recall 0.422


,method,K,confirmed_missed,confirmed_underpred,share_confirmed_any,median_gap_confirmed,min_auc_group,max_calib_shortfall,needs_trained_model,interpretable_rule
0,causal (proposed),22.00,8.0,0.0,0.364,0.399,0.704,0.318,False,True
1,protected only,5.00,1.0,0.0,0.200,0.435,0.705,0.318,False,True
2,slice search,22.00,13.0,0.0,0.591,0.431,0.676,0.401,True,True
3,clustering,18.00,7.0,0.0,0.389,0.360,0.595,0.385,True,False
4,confident learning,21.00,8.0,0.0,0.381,0.399,0.704,0.318,True,True
5,KNN disagreement,21.00,8.0,0.0,0.381,0.399,0.704,0.318,False,True
6,anomaly detection,21.00,8.0,0.0,0.381,0.399,0.704,0.318,False,True
7,random (mean of 20),20.65,8.0,0.0,0.388,0.399,0.704,0.318,False,True


## 8. The blind spot, side by side
For the recording-path groups where the O/E check is valid: what the test set says about the trained model
(calibration, missed positives) next to what the pre-training O/E check says about the labels. If the test
set looks clean while O/E is well below 1, the group's risk is in the *labels*.

The `rank_in_*` columns show where each baseline placed this exact group in its own full ranking
("3/25" = third of 25 groups it scored). A baseline that ranks a group 15th has not singled it out, even if
K is large enough to include it. Clustering is left out: its clusters are not rules, so the same group has
no rank there.

In [20]:
base_methods = ["slice search", "confident learning", "KNN disagreement", "anomaly detection"]
rp = conf[(conf.method == "causal (proposed)")].merge(oe[["group", "valid", "obs_over_exp", "ci_low", "ci_high"]], on="group")
rp = rp[rp.valid].copy()
for mth in base_methods:
    rk = RANKS[mth]
    rp[f"rank_in_{mth.split()[0].lower()}"] = [f"{rk.index(g) + 1}/{len(rk)}" if g in rk else "-" for g in rp.group]
blind = rp[["group", "share_train", "calib_ratio", "recall_gap", "confirmed_missed", "confirmed_underpred",
            "obs_over_exp", "ci_low", "ci_high"] + [c for c in rp.columns if c.startswith("rank_in_")]]
save(blind, f"{NB}_blind_spot.csv", show=0)
blind.round(3)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_blind_spot.csv  (6 rows x 13 cols)


,group,share_train,calib_ratio,recall_gap,confirmed_missed,confirmed_underpred,obs_over_exp,ci_low,ci_high,rank_in_slice,rank_in_confident,rank_in_knn,rank_in_anomaly
5,CholCheck == 0,0.283,1.153,0.377,True,False,0.726,0.629,0.834,49/184,19/22,19/22,11/22
6,AnyHealthcare == 0,0.207,0.980,0.097,False,False,1.105,0.984,1.238,93/184,16/22,16/22,4/22
13,Income <= 0.97,0.201,0.943,0.007,False,False,1.008,0.926,1.096,122/184,9/22,10/22,7/22
14,Income >= 4.8,0.200,1.011,0.199,True,False,0.941,0.844,1.046,84/184,14/22,14/22,22/22
15,RoutineCare == 0,0.154,1.022,0.329,True,False,0.739,0.603,0.897,76/184,20/22,20/22,3/22
16,Smoker == 1,0.448,0.993,-0.054,False,False,0.875,0.826,0.926,108/184,12/22,12/22,14/22


## 9. Ground truth (NHANES only): which methods found the groups whose diagnoses are really missed?
Only now is the blood-test truth opened. Every group was chosen in sections 3-5 from the training split's
*recorded* labels, so no method has seen the truth. The truth is scored on the **held-out adults**
(validation + test, `TRUTH_SCOPE = "heldout"`, the default): people whose records played no part in choosing
any group, so a group cannot look good by sharing chance noise with the data it was chosen from. This doubles
the missed cases compared with the test split alone. `"test"` and `"all"` (adds the training rows; more
power, not fully independent) are available as sensitivity checks.

For every group any method proposed:
* **missed share** = of the people in the group who truly have diabetes (diagnosed or HbA1c >= 6.5%), the
  share who were never told — the group's real label-error rate; **missed ratio** = group / everyone else;
* one-sided Fisher exact test: is the group's missed share higher than everyone else's?

**Answer key** = every single-attribute group (same construction as section 3) whose missed share is
significantly elevated (Holm over the whole universe).

**Short-list comparison (the main result).** The causal method's short list is the groups its O/E check
**flags** (valid and upper CI < 1) — the groups it would actually send for verification. Each baseline's
short list is its own top-m groups, m the same size; random draws m single-attribute groups (200 draws).
Two scores: the share of the list whose missed share is truly elevated (Bonferroni within the list), and the
list's mean missed ratio. **Bootstrap** (people resampled with replacement, 1,000 times) gives 95% intervals
for both, and for the difference "causal minus baseline" in mean missed ratio (an interval above 0 means the
causal list points to groups with more missed diagnoses).

Long-list scorecards (all K groups) are also saved, but are uninformative when K is close to the number of
possible groups — then every method, random included, covers the answer key.

In [21]:
from IPython.display import display
TRUTH_SCOPE = "heldout"   # "heldout" (val + test; recommended), "test", or "all"
N_BOOT = 1000

if TRUTH is None:
    print("No ground truth for this dataset (CDC). Section 9 runs for NHANES only.")
else:
    ev = {"all": pd.concat([train, val, test], ignore_index=True),
          "heldout": pd.concat([val, test], ignore_index=True),
          "test": test.copy()}[TRUTH_SCOPE]
    clu_ev = km.predict(sc_all.transform(ev[FEATURES].to_numpy(float)))
    def ev_mask(q):
        return clu_ev == int(q.split("__")[-1]) if q.startswith("__cluster__") else ev.eval(q).to_numpy(bool)
    true_pos = ev[TRUTH].to_numpy() == 1
    missed = true_pos & (ev[LABEL].to_numpy() == 0)
    print(f"scope '{TRUTH_SCOPE}': {len(ev):,} adults, {true_pos.sum():,} truly have diabetes, "
          f"{missed.sum():,} ({missed.sum() / true_pos.sum():.1%}) were never told")

    def miss_test(m, w=None):
        w = np.ones(len(m)) if w is None else w
        a = (w * (missed & m)).sum(); b = (w * (true_pos & m & ~missed)).sum()
        c = (w * (missed & ~m)).sum(); d = (w * (true_pos & ~m & ~missed)).sum()
        share_g, share_r = a / max(a + b, 1), c / max(c + d, 1)
        pval = stats.fisher_exact([[int(a), int(b)], [int(c), int(d)]], alternative="greater")[1] if (a + b) else np.nan
        return {"true_pos": int(a + b), "missed": int(a), "missed_share_group": share_g,
                "missed_share_rest": share_r, "missed_ratio": share_g / max(share_r, 1e-9), "p_truth": pval}

    # every proposed group
    gt = pd.concat([conf[["method", "group", "query"]].reset_index(drop=True),
                    pd.DataFrame([miss_test(ev_mask(q)) for q in conf["query"]])], axis=1)
    gt["p_truth_holm"] = np.nan
    for mth, g in gt.groupby("method"):
        ok = g["p_truth"].notna()
        gt.loc[g.index[ok], "p_truth_holm"] = multipletests(g.loc[ok, "p_truth"], method="holm")[1]
    gt["elevated"] = gt["p_truth_holm"] < ALPHA
    save(gt, f"{NB}_ground_truth.csv", show=0)

    # answer key
    uni = pd.DataFrame([{"group": n, "query": q, **miss_test(ev.eval(q).to_numpy(bool))} for n, q, _ in singles])
    uni["p_holm"] = multipletests(uni["p_truth"].fillna(1), method="holm")[1]
    uni = uni.sort_values("missed_ratio", ascending=False)
    key = uni[uni.p_holm < ALPHA]
    save(uni, f"{NB}_ground_truth_universe.csv", show=0)
    print(f"\nanswer key: {len(key)} of {len(uni)} single-attribute groups have a significantly elevated missed share")
    display(key[["group", "true_pos", "missed", "missed_share_group", "missed_share_rest", "missed_ratio"]].round(3))

    # long-list scorecard (kept for completeness)
    def covers(q_list, q_key):
        mk = ev.eval(q_key).to_numpy(bool)
        return any((mk & ev_mask(q)).sum() / max(min(mk.sum(), ev_mask(q).sum()), 1) > 0.7 for q in q_list)
    def gt_card(g, name):
        rec = np.mean([covers(list(g["query"]), qk) for qk in key["query"]]) if len(key) else np.nan
        return {"method": name, "K": len(g), "elevated": int(g.elevated.sum()),
                "precision": g.elevated.mean() if len(g) else np.nan, "recall_of_answer_key": rec}
    gsc = [gt_card(gt[gt.method == m], m) for m in order]
    rg = pd.DataFrame([gt_card(gt[gt.method == f"random#{r}"], "random") for r in range(20)])
    gsc.append({"method": "random (mean of 20)", **rg.drop(columns="method").mean().to_dict()})
    save(pd.DataFrame(gsc), f"{NB}_ground_truth_scorecard.csv", show=0)

    # ---------------- short lists ----------------
    flagged = list(oe.loc[oe.flag_under_recorded, "group"])
    if not flagged:
        flagged = list(oe[oe.valid].sort_values("obs_over_exp")["group"].head(2))
        print("no group flagged; using the 2 valid groups with the lowest O/E")
    m_short = len(flagged)
    q_of = {**{n: q for n, q, _ in singles + pairs}, **dict(zip(conf.group, conf["query"]))}
    lists = {"causal (proposed): O/E flags": flagged}
    for mth in ["slice search", "confident learning", "KNN disagreement", "anomaly detection"]:
        lists[mth] = RANKS[mth][:m_short]
    masks = {g: ev_mask(q_of[g]) for L in lists.values() for g in L}
    for n, q, m in singles:
        masks.setdefault(n, ev.eval(q).to_numpy(bool))

    def score_list(L, w=None):
        res = [miss_test(masks[g], w) for g in L]
        return (np.mean([r["p_truth"] < ALPHA / len(L) for r in res]), np.mean([r["missed_ratio"] for r in res]))

    rng2 = np.random.default_rng(SEED)
    rand_lists = [[singles[i][0] for i in rng2.choice(len(singles), m_short, replace=False)] for _ in range(200)]
    point = {k: score_list(L) for k, L in lists.items()}
    point["random (mean of 200)"] = tuple(np.mean([score_list(L) for L in rand_lists], axis=0))

    # bootstrap: resample people; the lists stay fixed (they were chosen without the truth)
    boot = {k: [] for k in lists}
    n_ev = len(ev)
    t0 = time.time()
    for b in range(N_BOOT):
        w = np.bincount(rng2.integers(0, n_ev, n_ev), minlength=n_ev).astype(float)
        for k, L in lists.items():
            boot[k].append(score_list(L, w))
    print(f"bootstrap ({N_BOOT}) took {time.time() - t0:.0f}s")
    B = {k: np.array(v) for k, v in boot.items()}     # [N_BOOT, 2]: share elevated, mean missed ratio
    ck = "causal (proposed): O/E flags"
    rows = []
    for k in list(lists) + ["random (mean of 200)"]:
        r = {"method": k, "m": m_short, "groups": "; ".join(lists.get(k, ["-"])),
             "share_truly_elevated": point[k][0], "mean_missed_ratio": point[k][1]}
        if k in B:
            r["share_ci"] = f"{np.percentile(B[k][:, 0], 2.5):.2f}-{np.percentile(B[k][:, 0], 97.5):.2f}"
            r["ratio_ci"] = f"{np.percentile(B[k][:, 1], 2.5):.2f}-{np.percentile(B[k][:, 1], 97.5):.2f}"
            if k != ck:
                diff = B[ck][:, 1] - B[k][:, 1]
                r["causal_minus_this_ratio_ci"] = f"{np.percentile(diff, 2.5):.2f} to {np.percentile(diff, 97.5):.2f}"
                r["share_boot_causal_higher"] = float((diff > 0).mean())
        rows.append(r)
    short = pd.DataFrame(rows)
    save(short, f"{NB}_ground_truth_shortlist.csv", show=0)
    print(f"\nshort lists (m = {m_short}), truth scope '{TRUTH_SCOPE}':")
    display(short.round(3))

    # does the pre-training O/E check point the right way? (causal candidates; O/E from TRAIN, truth from scope)
    chk = oe.merge(gt[gt.method == "causal (proposed)"][["group", "missed_ratio", "elevated"]], on="group")
    print("\nO/E (train, recorded labels only) vs true missed ratio:")
    display(chk[["group", "valid", "obs_over_exp", "ci_high", "flag_under_recorded", "missed_ratio", "elevated"]].round(3))
    v = chk[chk.valid]
    if len(v) >= 3:
        print(f"Spearman(O/E, missed ratio), valid groups only (n={len(v)}): {stats.spearmanr(v.obs_over_exp, v.missed_ratio)[0]:.2f}")

scope 'heldout': 12,310 adults, 2,002 truly have diabetes, 423 (21.1%) were never told
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_ground_truth.csv  (543 rows x 11 cols)
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_ground_truth_universe.csv  (22 rows x 9 cols)

answer key: 5 of 22 single-attribute groups have a significantly elevated missed share


,group,true_pos,missed,missed_share_group,missed_share_rest,missed_ratio
5,VisitedDoctor == 0,142,83,0.585,0.183,3.198
2,CholCheck == 0,216,94,0.435,0.184,2.362
4,RoutineCare == 0,138,61,0.442,0.194,2.276
11,GenHlth <= 2,338,106,0.314,0.191,1.646
3,AnyHealthcare == 0,298,84,0.282,0.199,1.417


wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_ground_truth_scorecard.csv  (8 rows x 5 cols)
bootstrap (1000) took 3s
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_ground_truth_shortlist.csv  (6 rows x 9 cols)

short lists (m = 3), truth scope 'heldout':


,method,m,groups,share_truly_elevated,mean_missed_ratio,share_ci,ratio_ci,causal_minus_this_ratio_ci,share_boot_causal_higher
0,causal (proposed): O/E flags,3,CholCheck == 0; RoutineCare == 0; Smoker == 1,0.667,1.834,0.67-0.67,1.59-2.10,NaN,NaN
1,slice search,3,GenHlth <= 2; Age <= 32; GenHlth <= 2 & Income...,0.333,1.396,0.33-0.67,1.10-1.69,0.09 to 0.84,0.992
2,confident learning,3,HeartDiseaseorAttack == 1; Stroke == 1; GenHlt...,0.000,0.487,0.00-0.00,0.39-0.58,1.09 to 1.63,1.000
3,KNN disagreement,3,HeartDiseaseorAttack == 1; Stroke == 1; GenHlt...,0.000,0.487,0.00-0.00,0.39-0.58,1.09 to 1.63,1.000
4,anomaly detection,3,Stroke == 1; HeartDiseaseorAttack == 1; Routin...,0.333,1.044,0.33-0.33,0.88-1.23,0.63 to 0.97,1.000
5,random (mean of 200),3,-,0.238,1.168,NaN,NaN,NaN,NaN



O/E (train, recorded labels only) vs true missed ratio:


,group,valid,obs_over_exp,ci_high,flag_under_recorded,missed_ratio,elevated
0,CholCheck == 0,True,0.726,0.834,True,2.362,True
1,RoutineCare == 0,True,0.739,0.897,True,2.276,True
2,Smoker == 1,True,0.875,0.926,True,0.862,False
3,Income >= 4.8,True,0.941,1.046,False,1.019,False
4,Income <= 0.97,True,1.008,1.096,False,1.038,False
5,AnyHealthcare == 0,True,1.105,1.238,False,1.417,True
6,Age <= 32,False,0.220,0.290,False,1.071,False
7,VisitedDoctor == 0,False,0.365,0.462,False,3.198,True
8,GenHlth <= 2,False,0.419,0.467,False,1.646,True
9,BMI <= 23.56,False,0.577,0.666,False,0.786,False


Spearman(O/E, missed ratio), valid groups only (n=6): -0.43


## 10. Extra short lists (NHANES only): clustering, protected-only, and two ablations
Added Nov 11, 2026 for the paper revision. Nothing above is changed; this section only reads what sections 3-9 built.

**Why.** Section 9 compares the method's flag list with slice search, confident learning, KNN and anomaly
detection. Three questions were left open:

1. **Clustering** was never scored as a short list. Its top-m clusters are scored here the same way.
2. **Protected-only audit**: all of its groups, scored the same way (it has no ranking, so the whole list is used).
3. **What does causal discovery add?** Two ablations of the proposed method:
   * *declared variable only* - the analyst's labeling-process variable with the O/E check, no discovery;
   * *O/E without the graph rule* - every candidate group whose O/E upper bound is below 1, valid or not.

**How.** Same scoring as section 9 (Bonferroni within each list, mean missed ratio), and the **same bootstrap
resamples of people** (same seed and the same order of random draws), so the first row reproduces the
section-9 interval for the proposed method and every "causal minus this" difference is paired.

**Two extra columns** describe what an auditor would get from each list:
* `missed_cases_in_list` - held-out missed diagnoses that fall inside the list's groups (union, no double counting);
* `missed_per_100_recorded_no` - missed diagnoses per 100 recorded "no" answers in the list: the yield per
  record checked. This is a different quantity from the missed *share* (which is per true case).

| Output | What it holds |
|---|---|
| `results/08_ground_truth_shortlist_extra.csv` | one row per extra list |
| `results/08_cluster_profiles.csv` | what the top clusters look like (a cluster is not a rule, so it is described) |

In [22]:
if TRUTH is None:
    print("No ground truth for this dataset (CDC). Section 10 runs for NHANES only.")
else:
    # ---- the extra lists (each chosen from the training split only; the truth is not used to choose) ----
    extra = {
        "causal (proposed): O/E flags": flagged,                                             # same list as section 9
        "clustering (top m)": list(cands.loc[cands.method == "clustering", "group"].head(m_short)),
        "protected only (all its groups)": list(cands.loc[cands.method == "protected only", "group"]),
        "ablation: declared variable only": [g for g in flagged if g.split()[0] in RECORDING],
        "ablation: O/E without the graph rule": list(oe.loc[oe.ci_high < 1, "group"]),
    }
    for L in extra.values():
        for g in L:
            masks.setdefault(g, ev_mask(q_of[g]))

    # ---- point estimates ----
    rec_no = ev[LABEL].to_numpy() == 0                                              # recorded "no": the labels that could be wrong
    print(f"all held-out adults: {100 * missed.sum() / rec_no.sum():.2f} missed diagnoses per 100 recorded 'no' answers")
    def list_detail(L):
        res = [miss_test(masks[g]) for g in L]
        return {"m": len(L), "groups": "; ".join(L),
                "true_cases_per_group": "; ".join(str(r["true_pos"]) for r in res),
                "missed_ratio_per_group": "; ".join(f"{r['missed_ratio']:.2f}" for r in res),
                "truly_elevated": int(sum(r["p_truth"] < ALPHA / len(L) for r in res)),
                "mean_missed_ratio": float(np.mean([r["missed_ratio"] for r in res])),
                # coverage: how many of the held-out missed diagnoses sit inside the list's groups (union, no double counting)
                "adults_in_list": int(np.any([masks[g] for g in L], axis=0).sum()),
                "missed_cases_in_list": int((missed & np.any([masks[g] for g in L], axis=0)).sum()),
                "share_of_all_missed": float((missed & np.any([masks[g] for g in L], axis=0)).sum() / missed.sum()),
                # yield per record checked: missed diagnoses per 100 recorded "no" answers in the list (all held-out adults: see printout)
                "missed_per_100_recorded_no": float(100 * (missed & np.any([masks[g] for g in L], axis=0)).sum()
                                                    / max((rec_no & np.any([masks[g] for g in L], axis=0)).sum(), 1))}

    # ---- bootstrap: the SAME resamples of people as section 9 (same seed, same order of random draws),
    #      so the first row reproduces the published interval and every difference is paired ----
    rng3 = np.random.default_rng(SEED)
    _ = [rng3.choice(len(singles), m_short, replace=False) for _ in range(200)]   # the 200 random lists drawn in section 9
    bootx = {k: [] for k in extra}
    t0 = time.time()
    for b in range(N_BOOT):
        w = np.bincount(rng3.integers(0, n_ev, n_ev), minlength=n_ev).astype(float)
        for k, L in extra.items():
            bootx[k].append(score_list(L, w)[1])                                   # mean missed ratio of the list
    BX = {k: np.array(v) for k, v in bootx.items()}
    print(f"bootstrap ({N_BOOT}) took {time.time() - t0:.0f}s")

    ck = "causal (proposed): O/E flags"
    rows_x = []
    for k, L in extra.items():
        r = {"list": k, **list_detail(L),
             "ratio_ci": f"{np.percentile(BX[k], 2.5):.2f}-{np.percentile(BX[k], 97.5):.2f}"}
        if k != ck:
            diff = BX[ck] - BX[k]
            r["causal_minus_this_ratio_ci"] = f"{np.percentile(diff, 2.5):.2f} to {np.percentile(diff, 97.5):.2f}"
            r["share_boot_causal_higher"] = float((diff > 0).mean())
        rows_x.append(r)
    short_x = pd.DataFrame(rows_x)
    save(short_x, f"{NB}_ground_truth_shortlist_extra.csv", show=0)
    print("check: the first row should match section 9's interval for the proposed method:",
          short.loc[short.method == ck, "ratio_ci"].iloc[0])
    display(short_x.round(3))

    # ---- what are the top clusters? (training split; a cluster is not a rule, so describe it) ----
    describe = [c for c in ["CholCheck", "RoutineCare", "VisitedDoctor", "AnyHealthcare", "Age", "BMI", "GenHlth"] if c in FEATURES]
    prof = [{"cluster": "everyone (training split)", "share_train": 1.0, "recorded_rate": float(y_tr.mean()),
             **{c: float(train[c].mean()) for c in describe}}]
    for g in extra["clustering (top m)"]:
        mc = km.labels_ == int(q_of[g].split("__")[-1])
        row = {"cluster": g, "share_train": float(mc.mean()), "recorded_rate": float(y_tr[mc].mean()),
               **{c: float(train.loc[mc, c].mean()) for c in describe}}
        for f in flagged:                                                           # overlap with the method's flagged groups
            row[f"share in [{f}]"] = float(train.eval(q_of[f]).to_numpy(bool)[mc].mean())
        prof.append(row)
    prof = pd.DataFrame(prof)
    save(prof, f"{NB}_cluster_profiles.csv", show=0)
    display(prof.round(3))

all held-out adults: 3.94 missed diagnoses per 100 recorded 'no' answers
bootstrap (1000) took 4s
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_ground_truth_shortlist_extra.csv  (5 rows x 14 cols)
check: the first row should match section 9's interval for the proposed method: 1.59-2.10


,list,m,groups,true_cases_per_group,missed_ratio_per_group,truly_elevated,mean_missed_ratio,adults_in_list,missed_cases_in_list,share_of_all_missed,missed_per_100_recorded_no,ratio_ci,causal_minus_this_ratio_ci,share_boot_causal_higher
0,causal (proposed): O/E flags,3,CholCheck == 0; RoutineCare == 0; Smoker == 1,216; 138; 997,2.36; 2.28; 0.86,2,1.834,7930,256,0.605,3.631,1.59-2.10,NaN,NaN
1,clustering (top m),3,cluster 2; cluster 4; cluster 9,56; 56; 22,0.84; 2.28; 2.85,2,1.990,2456,49,0.116,2.067,1.60-2.44,-0.55 to 0.20,0.208
2,protected only (all its groups),5,Sex == 1; Age <= 32; Age >= 67; race_black == ...,1025; 62; 709; 555; 564,0.94; 1.07; 0.81; 1.08; 1.17,0,1.014,10386,382,0.903,4.256,0.91-1.13,0.57 to 1.10,1.000
3,ablation: declared variable only,1,CholCheck == 0,216,2.36,1,2.362,3514,94,0.222,2.771,1.96-2.80,-0.81 to -0.27,0.000
4,ablation: O/E without the graph rule,7,CholCheck == 0; RoutineCare == 0; Smoker == 1;...,216; 138; 997; 62; 142; 338; 154,2.36; 2.28; 0.86; 1.07; 3.20; 1.65; 0.79,4,1.743,10459,315,0.745,3.351,1.55-1.95,-0.06 to 0.26,0.868


wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_cluster_profiles.csv  (4 rows x 13 cols)


,cluster,share_train,recorded_rate,CholCheck,RoutineCare,VisitedDoctor,AnyHealthcare,Age,BMI,GenHlth,share in [CholCheck == 0],share in [RoutineCare == 0],share in [Smoker == 1]
0,everyone (training split),1.000,0.128,0.717,0.846,0.834,0.793,49.807,29.265,2.751,NaN,NaN,NaN
1,cluster 2,0.104,0.039,0.996,1.000,1.000,0.988,45.793,26.223,1.948,0.004,0.0,0.234
2,cluster 4,0.047,0.032,0.210,0.000,0.278,0.181,38.306,29.174,3.033,0.790,1.0,0.413
3,cluster 9,0.049,0.025,0.231,0.000,0.315,0.188,38.691,27.255,2.694,0.769,1.0,0.595


### 10b. Per-record rates (added Nov 12, 2026)
The missed share in section 9 is **per true case**: of the adults who truly have diabetes, the share never told.
This cell adds the **per-record** view for the answer-key groups and the method's flagged groups: missed
diagnoses per 100 recorded "no" answers, in the group and among everyone else, and each group's true diabetes rate.
The two views can disagree: a group with few true cases can have a high missed share and a low per-record rate.

Output: `results/08_per_record_rates.csv` (read by notebook 09 for the paper's macros).

In [23]:
if TRUTH is None:
    print("No ground truth for this dataset (CDC). NHANES only.")
else:
    # Per-record yield for the answer-key groups and the method's flagged groups (held-out adults).
    # missed share            = missed / adults who truly have diabetes   (per true case; the paper's main measure)
    # missed per 100 "no"     = missed / adults whose recorded label is "no"   (per record checked)
    def per_record(name, m):
        g_no, r_no = rec_no & m, rec_no & ~m
        return {"group": name, "adults": int(m.sum()), "true_cases": int((true_pos & m).sum()),
                "true_rate_pct": 100 * (true_pos & m).sum() / max(m.sum(), 1),
                "missed": int((missed & m).sum()),
                "missed_share_pct": 100 * (missed & m).sum() / max((true_pos & m).sum(), 1),
                "missed_per_100_recorded_no": 100 * (missed & m).sum() / max(g_no.sum(), 1),
                "rest_missed_per_100_recorded_no": 100 * (missed & ~m).sum() / max(r_no.sum(), 1)}
    everyone = np.ones(len(ev), bool)
    pr_rows = [per_record("all held-out adults", everyone)]
    pr_rows[0]["rest_missed_per_100_recorded_no"] = np.nan
    names = list(dict.fromkeys(list(key["group"]) + list(flagged)))          # answer key first, then any other flagged group
    for g in names:
        masks.setdefault(g, ev_mask(q_of[g]))
        pr_rows.append({**per_record(g, masks[g]), "in_answer_key": g in set(key["group"]), "flagged_by_method": g in set(flagged)})
    per_rec = pd.DataFrame(pr_rows)
    save(per_rec, f"{NB}_per_record_rates.csv", show=0)
    display(per_rec.round(2))

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\08_per_record_rates.csv  (7 rows x 10 cols)


,group,adults,true_cases,true_rate_pct,missed,missed_share_pct,missed_per_100_recorded_no,rest_missed_per_100_recorded_no,in_answer_key,flagged_by_method
0,all held-out adults,12310,2002,16.26,423,21.13,3.94,NaN,NaN,NaN
1,VisitedDoctor == 0,2027,142,7.01,83,58.45,4.22,3.88,True,False
2,CholCheck == 0,3514,216,6.15,94,43.52,2.77,4.48,True,True
3,RoutineCare == 0,2016,138,6.85,61,44.20,3.15,4.12,True,True
4,GenHlth <= 2,4904,338,6.89,106,31.36,2.27,5.23,True,False
5,AnyHealthcare == 0,2606,298,11.44,84,28.19,3.51,4.07,True,False
6,Smoker == 1,5610,997,17.77,195,19.56,4.06,3.85,False,True
